# Merge Sampled Communities

Merges the sampled communities from several community-sampling sessions into one directory.

**Input:** a directory whose subdirectories are sampling sessions (the working
directory of each run). Each session contains:

- `labels.tsv`: one row per sampled community (first column `GraphID`);
- one `*_status.json` file;
- one subdirectory per sampled community, named after its `GraphID`.

**Output** (a new or empty directory, different from the input and not nested in it):

- every sampled community directory, copied as-is;
- every session's `*_status.json`, copied as-is;
- `labels.tsv`: the rows of all sessions' `labels.tsv` files under one header;
- `labels_extended.tsv`: `labels.tsv` plus a `StatusFile` column naming the
  `*_status.json` of the session each community comes from.

The input directory is only read, never modified. The notebook stops with an error,
before copying anything, if two sessions have a community or status file with the same
name, if the sessions' `labels.tsv` headers differ, or if
GraphIDs in a session's `labels.tsv` have no community directory. Subdirectories of a
session that have no row in its `labels.tsv` are reported and skipped.

## Setup

Requires the packages in `tools/requirements.txt`. Install them in a virtual environment
and select that environment as this notebook's kernel:

```
python -m venv .venv
.venv\Scripts\activate          # Linux/macOS: source .venv/bin/activate
pip install -r tools/requirements.txt
```

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

sns.set(style="darkgrid")

## Paths

In [ ]:
# Set these two paths.
INPUT_DIR = r""
OUTPUT_DIR = r""

In [ ]:
LABELS_FILENAME = "labels.tsv"
EXTENDED_LABELS_FILENAME = "labels_extended.tsv"
STATUS_FILE_GLOB = "*_status.json"
STATUS_FILE_COLUMN = "StatusFile"

In [ ]:
def resolve_dirs(input_dir, output_dir):
    if not input_dir or not output_dir:
        raise ValueError("Set both INPUT_DIR and OUTPUT_DIR.")

    in_dir = Path(input_dir).expanduser().resolve()
    out_dir = Path(output_dir).expanduser().resolve()

    if not in_dir.is_dir():
        raise NotADirectoryError(f"Input directory does not exist: {in_dir}")
    if in_dir == out_dir:
        raise ValueError("Input and output directories must be different.")
    if out_dir.is_relative_to(in_dir):
        raise ValueError(f"Output directory must not be inside the input directory: {out_dir}")
    if in_dir.is_relative_to(out_dir):
        raise ValueError(f"Input directory must not be inside the output directory: {in_dir}")
    if out_dir.exists():
        if not out_dir.is_dir():
            raise NotADirectoryError(f"Output path exists and is not a directory: {out_dir}")
        if any(out_dir.iterdir()):
            raise FileExistsError(f"Output directory is not empty: {out_dir}")

    return in_dir, out_dir


def read_labels(path):
    """Returns (header, rows) of a labels.tsv file; lines are kept verbatim (without newline)."""
    with open(path, "r", encoding="utf-8", newline="") as f:
        lines = [line.rstrip("\r\n") for line in f]
    lines = [line for line in lines if line.strip()]
    if not lines:
        raise ValueError(f"Empty labels file: {path}")
    return lines[0], lines[1:]


def scan_session(session_dir):
    """Collects and validates the status file, labels, and community dirs of one session."""
    status_files = sorted(p for p in session_dir.glob(STATUS_FILE_GLOB) if p.is_file())
    if len(status_files) != 1:
        raise ValueError(
            f"Expected exactly one {STATUS_FILE_GLOB} in session {session_dir}, "
            f"found {len(status_files)}: {[p.name for p in status_files]}")

    header, rows = read_labels(session_dir / LABELS_FILENAME)

    graph_ids = [row.split("\t", 1)[0] for row in rows]
    duplicates = sorted({g for g in graph_ids if graph_ids.count(g) > 1})
    if duplicates:
        raise ValueError(f"Duplicate GraphIDs in {session_dir / LABELS_FILENAME}: {duplicates}")

    missing = [g for g in graph_ids if not (session_dir / g).is_dir()]
    if missing:
        raise ValueError(
            f"{len(missing)} GraphID(s) in {session_dir / LABELS_FILENAME} have no community "
            f"directory in the session, e.g.: {missing[:5]}")

    # A directory without a labels row is not a complete community (e.g., a run cancelled
    # mid-serialization); it is reported and not copied.
    labeled = set(graph_ids)
    unlabeled = sorted(p.name for p in session_dir.iterdir() if p.is_dir() and p.name not in labeled)
    if unlabeled:
        print(
            f"WARNING: skipping {len(unlabeled)} directory(ies) in session {session_dir} that have "
            f"no row in its {LABELS_FILENAME}: {unlabeled}")

    return {
        "dir": session_dir,
        "status_file": status_files[0],
        "header": header,
        "rows": rows,
        "communities": graph_ids,
    }


def find_sessions(in_dir):
    session_dirs = sorted(
        p for p in in_dir.iterdir()
        if p.is_dir() and (p / LABELS_FILENAME).is_file())
    if not session_dirs:
        raise ValueError(f"No session directories (subdirectories with {LABELS_FILENAME}) in {in_dir}")
    return [scan_session(d) for d in session_dirs]


def check_no_collisions(sessions):
    headers = {s["header"] for s in sessions}
    if len(headers) != 1:
        raise ValueError(f"The sessions' {LABELS_FILENAME} headers differ: {sorted(headers)}")

    reserved = {LABELS_FILENAME, EXTENDED_LABELS_FILENAME}
    owner = {}
    for s in sessions:
        for name in [s["status_file"].name, *s["communities"]]:
            if name in reserved:
                raise ValueError(f"Session {s['dir']} has an entry named {name}, which the output reserves.")
            if name in owner:
                raise ValueError(f"Name collision: {name} exists in both {owner[name]} and {s['dir']}")
            owner[name] = s["dir"]

In [ ]:
in_dir, out_dir = resolve_dirs(INPUT_DIR, OUTPUT_DIR)
sessions = find_sessions(in_dir)
check_no_collisions(sessions)

print(f"Input:  {in_dir}")
print(f"Output: {out_dir}")
for s in sessions:
    print(f"  {s['dir'].name}: {len(s['communities']):,} communities, status file {s['status_file'].name}")
print(f"Total: {sum(len(s['communities']) for s in sessions):,} communities from {len(sessions)} session(s)")

In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)

header = sessions[0]["header"]
with open(out_dir / LABELS_FILENAME, "x", encoding="utf-8", newline="") as labels, \
     open(out_dir / EXTENDED_LABELS_FILENAME, "x", encoding="utf-8", newline="") as extended:
    labels.write(header + "\n")
    extended.write(f"{header}\t{STATUS_FILE_COLUMN}\n")

    for s in sessions:
        status_name = s["status_file"].name
        shutil.copy2(s["status_file"], out_dir / status_name)

        for graph_id in s["communities"]:
            shutil.copytree(s["dir"] / graph_id, out_dir / graph_id)

        for row in s["rows"]:
            labels.write(row + "\n")
            extended.write(f"{row}\t{status_name}\n")

        print(f"Merged {s['dir'].name}")

print(f"Done. Merged {sum(len(s['communities']) for s in sessions):,} communities into {out_dir}")

## Plots

The cells below read `labels_extended.tsv` from `OUTPUT_DIR`, so they can also be run on
their own on an existing merged directory: run the setup cells (install, imports, paths)
first, then the cells from here on.

In [ ]:
labels_df = pd.read_csv(Path(OUTPUT_DIR) / "labels_extended.tsv", sep="\t")
print(f"{len(labels_df):,} communities from {labels_df['StatusFile'].nunique()} session(s)")
labels_df[["NodeCount", "EdgeCount"]].describe()

### Node and edge counts per community

Swarm plots draw every community as a point; with many thousands of communities they get
slow and points may not all fit (seaborn then warns). Lower `SWARM_POINT_SIZE` if needed.

In [ ]:
SWARM_POINT_SIZE = 2

fig, (ax_nodes, ax_edges) = plt.subplots(1, 2, figsize=(10, 6))
sns.swarmplot(data=labels_df, y="NodeCount", size=SWARM_POINT_SIZE, ax=ax_nodes)
ax_nodes.set(title="Nodes per community", ylabel="Node count")
sns.swarmplot(data=labels_df, y="EdgeCount", size=SWARM_POINT_SIZE, color="C1", ax=ax_edges)
ax_edges.set(title="Edges per community", ylabel="Edge count")
ax_nodes.set(yscale="log")
ax_edges.set(yscale="log")
fig.tight_layout()
plt.show()

### Edge count vs. node count

In [ ]:
HEXBIN_GRID_SIZE = 40

fig, ax = plt.subplots(figsize=(7, 6))
hb = ax.hexbin(np.log10(labels_df["NodeCount"]), np.log10(labels_df["EdgeCount"]), gridsize=HEXBIN_GRID_SIZE, mincnt=1, cmap="viridis")
fig.colorbar(hb, ax=ax, label="Communities")
ax.set(title="Edge count vs. node count per community", xlabel="Node count (log10)", ylabel="Edge count (log10)")
# ax.set(xscale="log", yscale="log")
fig.tight_layout()
plt.show()

### Edge count vs. node count by session

Each community is colored by the `*_status.json` of its session. Overlapping hexbins of
several sessions would hide each other, so the joint panel is a scatter plot. The margins
show each session's node-count (top) and edge-count (right) distribution as density lines,
each normalized on its own so small sessions stay visible.

In [ ]:
g = sns.jointplot(
    data=labels_df,
    x="NodeCount",
    y="EdgeCount",
    hue="StatusFile",
    kind="scatter",
    height=8,
    joint_kws={"s": 12, "alpha": 0.6, "linewidth": 0},
    marginal_kws={"fill": False, "common_norm": False, "warn_singular": False, "cut": 0},
)
g.ax_joint.set(xscale="log", yscale="log")
g.set_axis_labels("Node count", "Edge count")
g.figure.suptitle("Edge count vs. node count per community, by session", y=1.02)
sns.move_legend(g.ax_joint, "upper left", bbox_to_anchor=(1.25, 1), title="Session status file")
plt.show()